# 🏀 Court scatter — a boro client linked to a quak DataTable

`CourtScatter` renders x/y shot positions on an NBA half court with a
**regl** WebGL point cloud (fast for hundreds of thousands of points),
colored by a categorical column.

It shares a `boro.Selection` with a `quak.DataTable`:

- **Filter the table → the court follows.** The court's `filter_by` is the
  shared selection, so it always shows the *current selection from other
  clients*.
- **Drag a box on the court → drill down locally.** Each box is `AND`-ed onto
  the court's *own* query only. It narrows the court further but never
  propagates back to the table (the court is wired `selection=(sel, None)` —
  filter-only).

In [1]:
import duckdb
import numpy as np

import boro
import quak
from court_scatter import CourtScatter

rng = np.random.default_rng(0)


def blob(n, mx, my, sx, sy):
    return np.c_[rng.normal(mx, sx, n), rng.normal(my, sy, n)]


# Synthetic NBA shot chart (coords in tenths of a foot, hoop at the origin).
zones = {
    "Restricted Area": (blob(40_000, 0, 15, 22, 22), 0.62, 2),
    "In The Paint":    (blob(20_000, 0, 90, 55, 45),  0.44, 2),
    "Mid-Range":       (blob(25_000, 0, 150, 130, 60), 0.40, 2),
    "Left Corner 3":   (blob(8_000, -228, 40, 12, 30), 0.39, 3),
    "Right Corner 3":  (blob(8_000, 228, 40, 12, 30),  0.39, 3),
    "Above Break 3":   (blob(30_000, 0, 250, 150, 40), 0.36, 3),
}
players = ["Curry", "Durant", "Jokic", "Doncic", "Tatum", "Embiid"]

xs, ys, zn, made, pts, plyr = [], [], [], [], [], []
for name, (coords, p_make, val) in zones.items():
    n = len(coords)
    xs.append(coords[:, 0])
    ys.append(coords[:, 1])
    zn += [name] * n
    made.append(rng.random(n) < p_make)
    pts.append(np.where(rng.random(n) < p_make, val, 0))
    plyr += list(rng.choice(players, n))

shots = duckdb.connect()
shots.register(
    "raw",
    {
        "x": np.concatenate(xs).astype("float32"),
        "y": np.concatenate(ys).astype("float32"),
        "zone": np.array(zn),
        "made": np.concatenate(made),
        "points": np.concatenate(pts).astype("int32"),
        "player": np.array(plyr),
    },
)
# Clip to the visible court and add a distance column.
shots.execute('''
    CREATE TABLE shots AS
    SELECT *, sqrt(x*x + y*y) / 10.0 AS distance_ft
    FROM raw
    WHERE x BETWEEN -250 AND 250 AND y BETWEEN -47.5 AND 422.5
''')
print(shots.table("shots").count("*").fetchone()[0], "shots")

126166 shots


## Share one selection between the table and the court

`selection=sel` on the table makes it a cross-filter source; `selection=(sel,
None)` on the court makes the court a **read-only consumer** — it reflects the
table's selection but can never write back to it.

In [2]:
coord = boro.Coordinator.connect(shots)
sel = boro.Selection.crossfilter(coord)

table = quak.DataTable(coord, "shots", selection=sel)
table

In [3]:
court = CourtScatter(
    coord,
    table="shots",
    x="x",
    y="y",
    color="zone",          # try "player" or "made"
    selection=(sel, None),  # filter-only: never disturbs the table
)
court

## Try it

- **Filter in the table above** (e.g. sort/select a player, or use a column
  brush) — the court instantly redraws to the current selection.
- **Drag a box on the court** to drill down. Drag again to narrow further
  (boxes stack as an intersection). Double-click, or the *clear drill-down*
  button, resets it.

The drill-down is a plain, Python-inspectable trait — and it stays local, so
the table above is untouched no matter how far you narrow.

In [ ]:
court.narrow          # the current drill-down stack: [[[x0, x1], [y0, y1]], ...]

In [ ]:
# Drive the drill-down from Python, too (e.g. isolate the left side near the rim):
court.push_narrow([[-250, 0], [-47.5, 150]])

In [ ]:
court.clear_narrow()   # local reset — the shared `sel` is never touched

In [ ]:
# The shared selection reflects only the table's clauses, not the court's boxes:
sel.value